# Produce the operating report
Gold is a compact report derived from cleaned Silver records. kWh per tonne is a ratio of total energy to total tonnes, not an unweighted average of row ratios.

In [ ]:
import re
# EDIT CATALOG if the detected default is not a writable Unity Catalog catalog.
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "iman_mining_lab"
VOLUME = "project_files"
for identifier in [CATALOG, SCHEMA, VOLUME]:
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier), "Use simple catalog/schema names"
assert CATALOG not in ["hive_metastore", "system"], "Choose a writable Unity Catalog catalog in CATALOG"
PREFIX = f"`{CATALOG}`.`{SCHEMA}`"
FILES = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
def table(name):
    assert re.fullmatch(r"[a-z_]+", name)
    return f"{PREFIX}.`{name}`"
print("Catalog:", CATALOG, "Schema:", SCHEMA, "Volume path:", FILES)


In [ ]:
from mining_core import machine_summary
source=spark.table(table("silver_shifts"))
assert source.count() <= 10000, "This exercise uses a small driver-side report"
summary=machine_summary(source.toPandas())
report=spark.createDataFrame([(str(z.machine_id),int(z.shifts),float(z.total_tonnes),float(z.total_energy_kwh),float(z.kwh_per_tonne)) for z in summary.itertuples(index=False)],"machine_id STRING, shifts LONG, total_tonnes DOUBLE, total_energy_kwh DOUBLE, kwh_per_tonne DOUBLE")
report.write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable(table("gold_machine_summary"))
display(report.orderBy("total_tonnes",ascending=False))
# Spark SQL is also used for the daily view.
display(spark.sql(f"SELECT shift_date, SUM(actual_tonnes) AS total_tonnes FROM {table('silver_shifts')} GROUP BY shift_date ORDER BY shift_date"))
assert report.count()==20


In [ ]:
import matplotlib.pyplot as plt
plot = report.toPandas().sort_values("total_tonnes",ascending=False)
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].bar(plot.machine_id,plot.total_tonnes);axes[0].set_title("Tonnes by machine");axes[0].tick_params(axis="x",rotation=90)
axes[1].bar(plot.machine_id,plot.kwh_per_tonne);axes[1].set_title("Energy intensity in kWh per tonne");axes[1].tick_params(axis="x",rotation=90)
plt.tight_layout();plt.show()


## Explain your results
Which machine moved the most tonnes? Which used the least energy per tonne? Why are those different questions? Work out whether the top throughput machine is also the most energy-efficient. Machine differences here are random synthetic variation, not evidence about real equipment quality.